# UMU-Bench track: text-only unlearning of UMU's released vanilla

**What this runs.** Start model = UMU-Bench's fine-tuned LLaVA-1.5-7B (`chengyewang/llava_smu_ft`). We unlearn the 50
forget people with **name-based questions only (no images)**, then ask the same facts by name, by portrait, and by
name + portrait. The leak is **G = portrait accuracy − name accuracy** on facts the vanilla knew both ways
(*doubly-correct*). Then we look inside the model at one checkpoint (Milestone J tool checks, logit lens).

**Where things go.** Everything is written to Google Drive `MyDrive/vlm_unlearn/`: unlearning runs in
`runs/llava/umu_unlearn/<run_id>/` (LoRA per checkpoint in `ckpt/step<N>/`), mechanism results in
`results/llava/mech_umu/<run_id>/step<N>/`. Every cell is **resumable**: after a disconnect, run section 1 again and
then the cell you were on.

**Order.** 1 setup (every session) → 2 vanilla check (done once) → 3 unlearning runs → 4 mechanism checks → 5 form sweep → 6 route-patching diagnostic → 7 causal input controls. The revised research question and results are in `proposal/proposal.tex` and `DEVIATIONS.md` DEV-011–013.
The old MLLMU-Bench main track (own vanilla fine-tuning, pilot, grid) is in `colab/archive_mllmu_track.ipynb`.

## 1. Setup (run every session)

In [ ]:
# --- set once ---------------------------------------------------------------
REPO_URL = "https://github.com/xuzhongwm/VLMUnlearn-private.git"
BRANCH = "main"
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VLMU_ROOT"] = "/content/drive/MyDrive/vlm_unlearn"   # checkpoints, logs, results
os.environ["VLMU_LOCAL_CACHE"] = "/content/vlmu_cache"            # fast local disk (UMU images, cached data)
os.environ["HF_HOME"] = "/content/hf_home"                        # base model weights (local disk)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["VLMU_ROOT"], exist_ok=True)

In [ ]:
from google.colab import userdata
import os, subprocess, tempfile
from pathlib import Path

def sync_repo():
    # In Colab Secrets, add GH_Token and enable notebook access. Never print the token.
    try:
        token = userdata.get("GH_Token")
    except Exception as exc:
        raise RuntimeError("Enable notebook access for GH_Token in Colab Secrets (key icon).") from exc
    if not token:
        raise RuntimeError("GH_Token is empty in Colab Secrets.")
    if not REPO_URL.startswith("https://github.com/") or "@" in REPO_URL:
        raise ValueError("REPO_URL must be a clean GitHub HTTPS URL without credentials.")
    repo = Path("/content/VLMUnlearn")
    if repo.exists() and not (repo / ".git").is_dir():
        raise RuntimeError(f"{repo} exists but is not a Git checkout; inspect it before retrying.")
    with tempfile.TemporaryDirectory() as tmp:
        askpass = Path(tmp) / "askpass.sh"
        askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*) printf '%s\n' x-access-token ;;\n  *Password*) printf '%s\n' \"$VLMU_GH_TOKEN\" ;;\n  *) exit 1 ;;\nesac\n")
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0", VLMU_GH_TOKEN=token)
        git = ["git", "-c", "credential.helper="]  # ignore stale cached credentials
        if repo.exists():
            subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)
            subprocess.run(git + ["-C", str(repo), "pull", "--ff-only", "origin", BRANCH], env=env, check=True)
        else:
            subprocess.run(git + ["clone", "-b", BRANCH, REPO_URL, str(repo)], env=env, check=True)

sync_repo()
%cd /content/VLMUnlearn
!git log --oneline -1


In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())"

## 2. Vanilla check: UMU's checkpoint scored the way UMU-Bench scores it (done; ~5 min on L4)
Generation + exact match on the forget set (100 facts × name / portrait) and the first 20 retain people.
Result 2026-11-05: forget name 0.75, portrait 0.81, both 0.65; AccF 0.823 = paper Table 1 (0.8233).
Only needed again if the readout code changes.

In [ ]:
!python scripts/eval_external.py --config configs/llava/vanilla.yaml --umu-protocol

## 3. Unlearning runs (name-based questions only, no images)
Each run gets its own folder (run id = method, lr, seed, optional tag) and evaluates at step 0 and every half or full epoch.
**How to read the final table** (all on doubly-correct facts, 1.00 at step 0):
- `forget dc txt` — asked by name: should fall (this is what we unlearn)
- `img` — asked by portrait; `t+i` — name + portrait; **`G` = img − txt** is the leak
- `retain dc txt / img` — other people: should stay near 1.00 (collateral damage)
- `degen` — share of the most common name-only answer: near 1.00 means the model says one thing to everything (collapse)

### 3a. Run 1: NPO only, lr 5e-5 (done)
Collapsed after epoch 1 (every name-only answer became "PAPAPA…"); only step 75 is usable. Run id `umu_npo_txt_lr5e-05_s0`.

In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo.yaml

### 3b. Run 2: NPO + retain (50 other people), lr 5e-5, 2 epochs (done; ~20 min on L4)
Step 150: forget name 0.63, portrait 0.97, name + portrait 0.97, **G 0.34**; retain name 0.69; no collapse.
Run id `umu_npo_rt_txt_lr5e-05_s0`. (The same config at lr 2e-5 barely forgot: `umu_npo_rt_txt_lr2e-05_s0`, stopped.)

In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo_rt.yaml --set train.lr=5.0e-5

### 3c. Run 3: stronger, broader retain (weight 3, 200 people), lr 5e-5 (done)
Step 150: forget name 0.92, portrait 0.97, G 0.05; retain name 0.96. Other people were protected, but the forgetting almost
vanished too. Run id `umu_npo_rt_txt_lr5e-05_s0_rw3n200`.


In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo_rt.yaml --set train.lr=5.0e-5 unlearn.retain_weight=3 umu.n_retain_unlearn=200 run_tag=rw3n200

### 3d. Run 4: same retain (weight 3, 200 people), lr 1e-4 (next; ~22 min on L4)
Question: is the damage to *other* people asked by name part of how this unlearning works, or just too little protection?
The readout now also has **seen** retain people (20 of the 200 the retain term trains on) next to the **unseen** 20.
- forget name falls, unseen retain falls, seen retain stays → the forgetting spreads to anyone asked by name; only people
  named in the retain set escape (it is the mechanism)
- forget name falls, both retain groups stay → person-specific forgetting is possible (run 3 was just too weak)
Run id `umu_npo_rt_txt_lr1e-04_s0_rw3n200`.

In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo_rt.yaml --set train.lr=1.0e-4 unlearn.retain_weight=3 umu.n_retain_unlearn=200 run_tag=rw3n200

## 4. Mechanism on one checkpoint (~5 min on L4)
Set the run and the step first (the step must be one of the run's evaluated checkpoints, e.g. 38 / 76 / 114 / 150).
Done so far: run 1 step 75, run 2 step 150.

In [ ]:
RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"   # a folder under runs/llava/umu_unlearn/
STEP = 150
CONFIG = "configs/llava/umu_npo_rt.yaml"   # supplies the UMU settings only; RUN_ID picks the run
LENS = f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}"
print(RUN_ID, STEP)

### 4a. Milestone J: are the hook tools exact on this model + LoRA?
On an L4 (bf16) the log ends with "MILESTONE J FAILED" because two subtraction checks, (ii) and (iii-b)[all], miss their 1%
tolerance by bf16 rounding (top-1 agreement ≈ 0.98). That is expected. Everything else must PASS, and
`txt/J1_token_groups` must show `n_name > 0`. On an A100, drop `--set mech.dtype=bfloat16` to run the full check in float32.

In [ ]:
!python scripts/check_hooks_umu.py --config {CONFIG} --run-id {RUN_ID} --step {STEP} --set mech.dtype=bfloat16

### 4b. Logit lens + figure
Per layer: how much the unlearning moved log p(correct answer), for name / portrait / name + portrait. Writes `lens.json` and
the figure `lens_figure_first.png` (also as PDF) under `LENS`. The second cell only redraws the figure.

In [ ]:
!python scripts/mech_umu_lens.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
!python scripts/plot_umu_lens.py --lens {LENS}/lens.json
from IPython.display import Image, display
display(Image(f"{LENS}/lens_figure_first.png"))

In [ ]:
!python scripts/plot_umu_lens.py --lens {LENS}/lens.json
from IPython.display import Image, display
display(Image(f"{LENS}/lens_figure_first.png"))

### 4c. Layer masking: H1 (late) or H1′ (early)? (~5 min on L4, no training)
Switches the unlearning LoRA off on some layers and checks whether the name-only answers come back.
Left plot: switch off one 4-layer window at a time. Right plot: switch off a whole band (0–9, 10–19, 20–31), and keep a band alone.
**Read:** if the late band (20–31) both restores the answers when off and produces the suppression on its own → **H1**;
if the early band (0–9) does → **H1′**. The log also prints generation accuracy on the facts the name route lost.

In [ ]:
!python scripts/mech_umu_layer_mask.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
from IPython.display import Image, display
display(Image(f"{LENS}/layer_mask.png"))

### 4d. Where on the input does the edit act? ||ΔW x|| by token group (~5 min on L4, no training)
For every LoRA module, the size of the unlearning change on each token, averaged over: **name** tokens, **image** tokens,
the rest of the prompt (**template**) and the **last** prompt token. (a) name-only questions, (b) portrait questions,
(c) name tokens with vs without the portrait and for retain people, (d) the change written along the answer direction at the last token.
**Read:** name ≫ template in layers 0–19 → the edit is bound to the name tokens (H1′). Name tokens equally hit with the portrait
and for retain people → the portrait route bypasses the edit, and every name pays the collateral.

In [ ]:
RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"; STEP = 150; CONFIG = "configs/llava/umu_npo_rt.yaml"
LENS = f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}"
!python scripts/mech_umu_delta_wx.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
from IPython.display import Image, display
display(Image(f"{LENS}/delta_wx.png"))

### 4e. Which token positions carry the edit? Position masking (~8 min on L4, no training)
Switches the unlearning LoRA off (or keeps it on) at chosen tokens — name / last prompt token / the rest of the prompt — in every
layer, and checks how much of the forgetting on name-only questions comes back. Exact switch (checked at the start of the log:
both differences should be 0). Left plot: LoRA off at those tokens (necessary?). Right plot: LoRA on only at those tokens (sufficient?).
**Read:** name → the edit is in how the name is processed; last → in how the last position reads the identity; both needed →
the name-to-last-token pathway as a whole. (Re-run 4d first if you want the new relative ||ΔW x|| / ||W x|| column.)

In [ ]:
RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"; STEP = 150; CONFIG = "configs/llava/umu_npo_rt.yaml"
LENS = f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}"
!python scripts/mech_umu_position_mask.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
from IPython.display import Image, display
display(Image(f"{LENS}/position_mask.png"))

## 5. Query-form component sweep (DEV-013; ~15 min on L4, no training)
Which part of the question does the unlearning key on? Starts from the training form ("USER: {question}\nASSISTANT:", no image,
no options) and changes one component at a time: **A** image context (own portrait / blank image / another person's portrait /
576 filler text tokens), **B** how the person is named, **C** wording (paraphrases), **D** answer format (option list, A–D,
fill-in-the-blank), **E** chat frame. Same 91 doubly-correct facts throughout.
**Read the bars:** recovery R = how much less the answer is suppressed than in the training form (0 = just as forgotten,
1 = not forgotten at all). The log ends with the pre-registered predictions P1–P5 marked PASS / FAIL (thresholds in DEVIATIONS DEV-013).
`--no-gen` skips the generation check (~6 min faster).

In [ ]:
RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"; STEP = 150; CONFIG = "configs/llava/umu_npo_rt.yaml"
LENS = f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}"
!python scripts/mech_umu_form_sweep.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
from IPython.display import Image, display
display(Image(f"{LENS}/form_sweep.png"))

## 6. Why does a blank image bypass forgetting? Causal route patching

This uses the **same name question and answer** with and without a 336×336 blank image. If section 5 generation files are present, it selects facts answered incorrectly without an image and correctly with a blank image; otherwise it uses doubly-correct forget facts in stable order. At each decoder layer, it swaps only the state at the **last prompt token before the answer** between the two routes, in both directions. It repeats the swap with the unlearning LoRA disabled as a vanilla control.

The plot shows the patch effect on log p of the **first correct answer token**, after subtracting the same effect in vanilla. A negative text → blank effect means the image route becomes more suppressed; a positive blank → text effect means the text route recovers. These are teacher-forced causal readouts, not generated-answer accuracy. The script saves per-fact values, baseline scores, and bootstrap intervals to `blank_patch.json`.

The default runs 12 facts on a coarse layer grid. If a band looks important, rerun with `--layers all --max-facts 0` for all eligible facts and every layer. The full run is substantially slower.


In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Image, display

RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"
STEP = 150
CONFIG = "configs/llava/umu_npo_rt.yaml"
REPO = Path("/content/VLMUnlearn")
SCRIPT = REPO / "scripts/mech_umu_blank_patch.py"
if not SCRIPT.is_file():
    head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    raise FileNotFoundError(f"{SCRIPT} is missing in Colab checkout {head.stdout.strip()}. Re-run the setup/sync cell; the script was added in commit 977a294.")
subprocess.run(["python", str(SCRIPT), "--config", CONFIG, "--run-id", RUN_ID, "--step", str(STEP)], cwd=REPO, check=True)
figure = Path(f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}/blank_patch.png")
if not figure.is_file():
    raise FileNotFoundError(f"The script finished but did not create {figure}; inspect blank_patch_log.txt in the same folder.")
display(Image(filename=str(figure)))


## 7. What input feature makes a blank image bypass text unlearning?

This is the first causal input-control experiment in the revised proposal. It holds the **same name question, person, fact, and answer** fixed and evaluates all frozen doubly-correct forget and retain facts. It compares no image, blank image before/after the question, a blank image whose projected visual features are set to zero **without removing its image-token slots**, and neutral text prefixes/suffixes of several lengths. Both the unlearned model and adapter-off vanilla are scored.

Read both **generation keyword hit** and **correct-answer log probability**; the long-text control previously disagreed across these readouts. `R=0` means suppression as strong as the no-image base, and `R=1` means no unlearning effect. The zero-projection condition is out of distribution: interpret it only if vanilla still answers correctly. The run is resumable and saves per-item JSONL plus `summary.json` to Drive. It does **not** yet block image-token attention; that intervention depends on which contrast this experiment identifies.

Run section 1 after any Colab restart. The script must exist in the synced checkout. For a quick plumbing check, add `--max-facts 4 --no-gen`; then rerun the default cell for the full experiment.


In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd

RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"
STEP = 150
CONFIG = "configs/llava/umu_npo_rt.yaml"
REPO = Path("/content/VLMUnlearn")
SCRIPT = REPO / "scripts/mech_umu_input_cues.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(f"{SCRIPT} is missing. Sync the private repo in section 1 first.")
subprocess.run(["python", str(SCRIPT), "--config", CONFIG, "--run-id", RUN_ID, "--step", str(STEP)], cwd=REPO, check=True)
RESULT = Path(f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}/input_cues/summary.json")
data = json.loads(RESULT.read_text())["summary"]
for group in ("forget", "retain"):
    print(group)
    display(pd.DataFrame([{"form": form, "n": v["n_facts"], "tokens": round(v["prompt_tokens_mean"], 1),
                           "R": round(v["recovery_R"], 2), "gen vanilla": v["generation_keyword_0"],
                           "gen unlearned": v["generation_keyword_u"]}
                          for form, v in data[group].items()]))
